In [ ]:
# 09_leave_domain_out — Cell 1: random split vs domain-held-out split (TF-IDF + LR), c2020 & c2025
import os, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"; os.makedirs(R, exist_ok=True)
SEED = 0

CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}
for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:1500]
    df["domain"] = df["domain"].fillna("unk")

def ev(y, p, s):
    return dict(bacc=round(balanced_accuracy_score(y, p), 3),
                f1=round(f1_score(y, p, average="macro"), 3),
                auc=round(roc_auc_score(y, s), 3))

def fit_eval(tr, te, tag):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    clf.fit(v.fit_transform(tr["text"]), tr["label"])
    s = clf.decision_function(v.transform(te["text"]))
    p = (s > 0).astype(int)
    return dict(split=tag, n_train=len(tr), n_test=len(te),
                n_domains_train=tr["domain"].nunique(), n_domains_test=te["domain"].nunique(),
                **ev(te["label"], p, s))

rows = []
for name, df in CORP.items():
    # (a) random 70/30 stratified — domains free to repeat between train/test
    tr, te = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SEED)
    rows.append(dict(corpus=name, **fit_eval(tr, te, "random_split")))

    # (b) domain-held-out — GroupShuffleSplit guarantees zero domain overlap
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
    tr_idx, te_idx = next(gss.split(df, groups=df["domain"]))
    tr, te = df.iloc[tr_idx], df.iloc[te_idx]
    overlap = len(set(tr["domain"]) & set(te["domain"]))
    rows.append(dict(corpus=name, **fit_eval(tr, te, "domain_heldout"), domain_overlap=overlap))

res = pd.DataFrame(rows)
res.to_csv(f"{R}/leave_domain_out.csv", index=False)
print(res.to_string(index=False))